In [ ]:
from Classes.MyPDFReader import MyPDFReader
from Classes.Chunker import Chunker
from Classes.EmbeddingStore import EmbeddingStore
from Classes.Retriever import Retriever
from Classes.ModelBuilder import ModelBuilder
from Classes.WebRAGEngine import WebRAGEngine
from Classes.LocalRAGEngine import LocalRAGEngine
from Classes.MyAgent import MyAgent
from Classes.ParamsLoader import ParamsLoader
from Classes.PredictionGenerator import PredictionGenerator

In [2]:
params = ParamsLoader.load_params("Utils/params.yaml")
params

{'llm_model_name': 'Qwen/Qwen3-1.7B',
 'embedder_model_name': 'Qwen/Qwen3-Embedding-0.6B',
 'pdf_name': 'rulebook.pdf',
 'Chunker': {'chunk_size': 1000, 'overlap': 200},
 'MyAgent': {'retrieval_threshold': 0.4, 'local_top_k': 5, 'web_top_k': 5}}

In [3]:
embed_model = ModelBuilder.build_embedder(params['embedder_model_name'])
mypdfreader = MyPDFReader(params['pdf_name'])
chunker = Chunker(mypdfreader, **params['Chunker'])
embedding_store = EmbeddingStore(chunker, embed_model)
retriever = Retriever(embedding_store, embed_model)
model = ModelBuilder.build_llm(params['llm_model_name'])
local_rag_engine = LocalRAGEngine(retriever, model)
web_rag_engine = WebRAGEngine(model)
agent = MyAgent(local_rag_engine, web_rag_engine, **params['MyAgent'])

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

Some parameters are on the meta device because they were offloaded to the disk.


In [6]:
pred_generator = PredictionGenerator(agent)
pred_generator.run('Utils/eval_dataset.json', 'Utils/eval_predictions.json')

[{'id': 1,
  'question': 'We attacked a monster and reduced it to 0 HP. Then we applied retaliate damage to the attacker. Did we play this correctly?',
  'expected_was_played_correctly': False,
  'expected_category': 'Combat',
  'prediction': {'explanation': 'The user did not play the retaliate correctly.',
   'was_played_correctly': False,
   'category': 'Combat',
   'source_type': 'rulebook',
   'sources': [{'page_number': 20, 'chunk_number': 3, 'score': 0.607},
    {'page_number': 26, 'chunk_number': 2, 'score': 0.595},
    {'page_number': 26, 'chunk_number': 3, 'score': 0.586},
    {'page_number': 4, 'chunk_number': 2, 'score': 0.581},
    {'page_number': 28, 'chunk_number': 4, 'score': 0.578}],
   'best_retrieval_score': 0.607}},
 {'id': 2,
  'question': 'A character played the top action of one card and the bottom action of another card on the same turn. Did we play this correctly?',
  'expected_was_played_correctly': True,
  'expected_category': 'Character',
  'prediction': {'ex